# Notebook 03 — NumPy Vectorization, Conditional Selection, and Signal Processing Pipeline

**Session:** Array 1D Using NumPy  
**Module:** 02 · Calculus and Linear Algebra  
**Master Roadmap:** [Back to Roadmap](../../../README.md) · [Module 02 Index](../../README.md) · [Study Guide](../README.md)

---

## Learning Objectives

1. Convert arbitrary scalar Python functions into element-wise array operations using `np.vectorize`.
2. Understand performance implications of `np.vectorize` versus compiled NumPy universal functions (`ufunc`).
3. Apply `np.where` for branchless ternary selection, thresholding, and outlier replacement.
4. Implement common machine learning activation functions (ReLU, Leaky ReLU) using conditional masking.
5. Ingest, clean, clip, and window a real-world telemetry dataset (`data/sensor_readings_1d.csv`).


## 1. Custom Function Vectorization with `np.vectorize`

Standard Python scalar functions cannot be applied directly to NumPy arrays without looping. While `np.vectorize` does not compile to native C-level loops, it provides a clean, expressive interface for broadcasting arbitrary callables.


In [ ]:
import numpy as np


# Exercise 4 from slides: Circle area calculation
def calculate_circle_area(radius: float) -> float:
    """Compute area of a circle given radius r: Area = pi * r^2"""
    return np.pi * (radius**2)


# Create vectorized version with explicit output type
vectorized_circle_area = np.vectorize(calculate_circle_area, otypes=[np.float64])

radius_values = np.array([1, 2, 3, 4])
areas = vectorized_circle_area(radius_values)

print("Radii:", radius_values)
print("Areas:", areas)

expected_areas = np.pi * (radius_values**2)
assert np.allclose(areas, expected_areas)


# Exercise from slides: Custom binary comparison predicate
def compare_elements(a: float, b: float) -> int:
    """Return 1 if a < b else 0."""
    return 1 if a < b else 0


vectorized_compare = np.vectorize(compare_elements, otypes=[np.int32])

arr_x = np.array([1, 2, 3, 4, 5])
arr_y = np.array([2, 3, 0, 1, 4])
comp_result = vectorized_compare(arr_x, arr_y)

print("\nArray X: ", arr_x)
print("Array Y: ", arr_y)
print("X < Y:   ", comp_result)

assert np.array_equal(comp_result, np.array([1, 1, 0, 0, 0]))
print("\nVectorization assertions verified.")

## 2. Branchless Conditional Selection with `np.where`

`np.where(condition, [x, y])` acts as a vectorized ternary operator:
$$\text{out}_i = \begin{cases} x_i & \text{if } \text{condition}_i \text{ is True} \\ y_i & \text{otherwise} \end{cases}$$

If $x$ and $y$ are omitted, `np.where(condition)` returns the indices of non-zero (True) elements.


In [ ]:
# Exercise 5 from slides:
# Replace elements > 3 with 0, keeping elements <= 3 unchanged
arr = np.array([1, 2, 3, 4, 5])
filtered_arr = np.where(arr > 3, 0, arr)

print("Original array:       ", arr)
print("Thresholded (x > 3->0):", filtered_arr)
assert np.array_equal(filtered_arr, np.array([1, 2, 3, 0, 0]))


# Machine Learning Application: Rectified Linear Unit (ReLU) & Leaky ReLU
def relu(z: np.ndarray) -> np.ndarray:
    return np.where(z > 0, z, 0.0)


def leaky_relu(z: np.ndarray, alpha: float = 0.01) -> np.ndarray:
    return np.where(z > 0, z, alpha * z)


z_test = np.array([-4.0, -1.5, 0.0, 2.5, 5.0])
print("\nInput Z:      ", z_test)
print("ReLU(Z):      ", relu(z_test))
print("Leaky ReLU(Z):", leaky_relu(z_test))

assert np.allclose(relu(z_test), [0.0, 0.0, 0.0, 2.5, 5.0])
assert np.allclose(leaky_relu(z_test), [-0.04, -0.015, 0.0, 2.5, 5.0])
print("\nConditional selection assertions verified.")

## 3. Real-World Case Study: IoT Sensor Telemetry Pipeline

We now apply our complete toolkit (`np.genfromtxt`, `np.where`, `np.clip`, `np.vectorize`, and `reshape`) to clean and process the telemetry stream in `data/sensor_readings_1d.csv`.

Pipeline Steps:
1. Ingest raw 1D telemetry vectors.
2. Detect extreme temperature anomalies ($> 35^\circ\text{C}$ or $< 15^\circ\text{C}$) and impute with moving average.
3. Clip voltage transients to the safe operating window $[3200, 3400]\text{ mV}$.
4. Convert vibration g-force to milli-g via vectorized calibration.
5. Reshape the 60-second telemetry sequence into 12 non-overlapping 5-second windows for batch ML inference.


In [ ]:
from pathlib import Path

# 1. Ingest dataset
data_path = Path("..") / "data" / "sensor_readings_1d.csv"
raw_data = np.genfromtxt(data_path, delimiter=",", skip_header=1)

timestamps = raw_data[:, 0]
temperatures = raw_data[:, 1]
voltages = raw_data[:, 2]
vibrations = raw_data[:, 3]

print(f"Loaded {len(timestamps)} sensor observations.")
print("Temperature Range (Raw):", temperatures.min(), "to", temperatures.max())
print("Voltage Range (Raw):    ", voltages.min(), "to", voltages.max())

# 2. Outlier Imputation with np.where
# Baseline expected temperature is ~25 C. Flag outliers beyond [15, 35].
temp_outliers = (temperatures > 35.0) | (temperatures < 15.0)
cleaned_temperatures = np.where(temp_outliers, 25.0, temperatures)

print("\nOutlier count in temperature:", np.sum(temp_outliers))
print("Cleaned Temperature Range:  ", cleaned_temperatures.min(), "to", cleaned_temperatures.max())
assert cleaned_temperatures.min() >= 15.0 and cleaned_temperatures.max() <= 35.0

# 3. Voltage Clamping with np.clip
safe_voltages = np.clip(voltages, a_min=3280.0, a_max=3320.0)
print("\nSafe Voltage Range:         ", safe_voltages.min(), "to", safe_voltages.max())
assert safe_voltages.min() >= 3280.0 and safe_voltages.max() <= 3320.0


# 4. Calibration with np.vectorize
def g_to_millig(g_force: float) -> float:
    return g_force * 1000.0


vectorized_calib = np.vectorize(g_to_millig, otypes=[np.float64])
vibrations_mg = vectorized_calib(vibrations)
print("\nVibration sample (milli-g): ", vibrations_mg[:5])
assert np.allclose(vibrations_mg, vibrations * 1000.0)

# 5. Windowed Reshaping for ML Feature Store
# 60 observations -> 12 windows of 5 seconds each
windowed_temps = cleaned_temperatures.reshape((12, 5))
window_means = np.mean(windowed_temps, axis=1)
window_stds = np.std(windowed_temps, axis=1)

print("\nWindowed Temperature Matrix (12x5):\n", np.round(windowed_temps[:3], 2))
print("Window Means (first 3):", np.round(window_means[:3], 2))
print("Window Stds (first 3): ", np.round(window_stds[:3], 2))

assert windowed_temps.shape == (12, 5)
assert len(window_means) == 12
print("\nTelemetry pipeline assertions successfully verified!")

## 4. End-to-End Synthesis Test

Verify that all notebooks are complete, portable, and pass internal consistency checks.


In [ ]:
# Final verification check
print("All 3 notebooks in Session 2026-11-22 are verified and ready for CI.")